# Sparse Feature Interpretation

Once you've trained an SAE, the next step is understanding **what each
feature represents**. This notebook demonstrates tools for interpreting
individual SAE features: what activates them, how they interact, and
what they do to model outputs.

This notebook covers the `irtk.sparse_features` module.

## Why This Matters

Sparse feature analysis examines the features discovered by sparse autoencoders — their activation patterns, correlations with other features, and downstream effects on predictions. This is how you determine whether SAE features are genuinely interpretable.

**Key references:**
- [Bricken et al. (2023) "Towards Monosemanticity"](https://transformer-circuits.pub/2023/monosemantic-features/index.html) — Sparse autoencoders find interpretable features
- [Cunningham et al. (2023) "Sparse Autoencoders Find Highly Interpretable Features"](https://arxiv.org/abs/2309.08600) — SAE features in larger language models

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt

import irtk
from irtk import sparse_features
from irtk.sae import SparseAutoencoder, train_sae

model = irtk.HookedTransformer.from_pretrained("gpt2")
tokenizer = model.tokenizer
print(f"Model: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## 0. Train a Quick SAE

Train a small SAE on residual stream activations for analysis.

In [ ]:
# Collect activations from a few prompts
hook = "blocks.6.hook_resid_post"
prompts = [
    "The Eiffel Tower is located in",
    "The capital of France is",
    "London is the capital of",
    "Berlin is a city in",
    "Tokyo is the capital of",
    "The weather today is very",
    "My favorite food is definitely",
    "The president of the United States",
]
token_seqs = [model.to_tokens(p) for p in prompts]

all_acts = []
for tokens in token_seqs:
    _, cache = model.run_with_cache(tokens)
    all_acts.append(np.array(cache[hook]))
activations = jnp.concatenate(all_acts, axis=0)
print(f"Collected {activations.shape[0]} activation vectors")

# Train SAE
result = train_sae(activations, d_model=model.cfg.d_model, n_features=512,
                   l1_coeff=5e-4, epochs=20, lr=3e-4)
sae = result.sae
print(f"SAE: {sae.d_model}d -> {sae.n_features} features")

## 1. Feature Activation Examples

Find the top token contexts that maximally activate a feature.

In [ ]:
# Pick a feature with high activation
feat_acts = sae.encode(activations)
mean_acts = np.array(feat_acts.mean(axis=0))
top_feature = int(np.argmax(mean_acts))
print(f"Analyzing feature {top_feature} (mean activation: {mean_acts[top_feature]:.4f})")

examples = sparse_features.feature_activation_examples(
    sae, model, token_seqs, top_feature, hook, k=10
)

print(f"\nTop activating contexts:")
for ex in examples[:5]:
    toks = [tokenizer.decode([int(t)]) for t in ex['tokens']]
    pos = ex['position']
    print(f"  act={ex['activation']:.3f}  pos={pos}  token='{toks[pos]}'  context: {''.join(toks[:pos+1])}")

## 2. Feature Token Bias

Which vocabulary tokens most strongly activate a feature through
the embedding matrix?

In [ ]:
bias = sparse_features.feature_token_bias(sae, model, top_feature, k=10)

print(f"Feature {top_feature} - top responding tokens:")
print("\nMost activated by:")
for tid, score in bias['top_positive']:
    print(f"  {tokenizer.decode([tid]):>15s}: {score:.4f}")

print("\nMost suppressed by:")
for tid, score in bias['top_negative']:
    print(f"  {tokenizer.decode([tid]):>15s}: {score:.4f}")

## 3. Feature Downstream Effect

What output tokens does this feature promote or suppress?

In [ ]:
tokens = model.to_tokens("The Eiffel Tower is located in")
effect = sparse_features.feature_downstream_effect(
    sae, model, tokens, top_feature, hook, k=10
)

print(f"Feature {top_feature} downstream effect (logit diff norm: {effect['logit_diff_norm']:.4f}):")
print("\nTokens promoted:")
for tid, change in effect['top_promoted'][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {change:+.4f}")

print("\nTokens suppressed:")
for tid, change in effect['top_suppressed'][:5]:
    print(f"  {tokenizer.decode([tid]):>15s}: {change:+.4f}")

## 4. Feature-to-Feature Correlation

Do features co-activate? This reveals feature clusters.

In [ ]:
# Find second most active feature
sorted_feats = np.argsort(mean_acts)[::-1]
feat_a = int(sorted_feats[0])
feat_b = int(sorted_feats[1])

corr = sparse_features.feature_to_feature_correlation(
    sae, model, token_seqs, feat_a, feat_b, hook
)
print(f"Features {feat_a} and {feat_b}:")
print(f"  Correlation: {corr['correlation']:.4f}")
print(f"  Co-activation rate: {corr['co_activation_rate']:.4f}")
print(f"  P({feat_a} active | {feat_b} active): {corr['conditional_a_given_b']:.4f}")
print(f"  P({feat_b} active | {feat_a} active): {corr['conditional_b_given_a']:.4f}")

## Summary

| Function | Purpose |
|----------|--------|
| `feature_activation_examples()` | Top-k token contexts for a feature |
| `feature_to_feature_correlation()` | Co-activation between feature pairs |
| `feature_circuit()` | Trace upstream components driving a feature |
| `feature_token_bias()` | Vocabulary tokens a feature responds to |
| `feature_downstream_effect()` | Output tokens promoted/suppressed by feature |